# Reproducing Shi, Apker and Buneo (2013) on CRCNS ppc-1

**Paper.** Shi Y, Apker G, Buneo CA (2013). Multimodal representation of limb endpoint position in
the posterior parietal cortex. *J Neurophysiol* 109: 2097-2107.
[doi:10.1152/jn.00223.2012](https://doi.org/10.1152/jn.00223.2012)

**Dataset.** CRCNS ppc-1, Buneo, Shi, Apker and VanGilder (2016),
[doi:10.6080/K0CZ353K](https://doi.org/10.6080/K0CZ353K), identified here as
`crcns:ppc-1/unversioned`. It is read from the public bucket
(`https://crcnsarchive.s3-us-west-2.amazonaws.com/ppc-1/live/data.tar.gz`, streamed, not
downloaded).

**Version.** CRCNS data sets are not versioned, and ppc-1 has one published copy, under `live/`
(no `old/` copy exists in the bucket), so that copy is the one pinned. The paper cites no version.
To detect a later change: `data.tar.gz` has MD5 `bbb341b8947a6314b2618f0caa0b64f8` in the data
set's `checksums.md5` (printed below as read), size 4,394,813,551 bytes; data descriptor version 0.8
(Dec 7, 2016).

**Claims.** `claims/shi-2013.yaml`, written from the version of record (Methods, Results,
Discussion, Table 1, legends of Figures 1 to 6; the paper has no supplement and links no code), and
checked by a second reader.

**What the data set holds, in short.** Spike times of sorted units and strobed event codes for
each session, in MATLAB and Plexon files; each trial's events end with a text record that gives the
target, a two-valued condition code and the hand position. All sessions have monkey X's target
layout; monkey B is not in the data set. Which condition code is Vision is not documented and is
inferred from the data (see "Which code is Vision"). Outcomes that depend on it are marked as
resting on that assumption.

| Claim | Status |
|---|---|
| behavior-mean-position-sessions | tested |
| behavior-variability-sessions | tested |
| table1-endpoint-means | tested |
| table1-endpoint-sd | tested |
| fig3-example-neuron | tested |
| movement-epoch-tuning | tested |
| hold-position-main-effect | tested |
| hold-vision-main-effect | tested |
| hold-interaction | tested |
| hold-subepochs | tested |
| hold-overall-effects | tested |
| correlation-endpoint-datasets | tested |
| fig4-indx1-negative | tested |
| fig4-indx1-means | tested |
| indx1-tuning-shift | tested |
| indx2-means | tested |
| rate-variance-nv-higher | tested |
| fig5-fano-factor | tested |
| cv2-larger-nv | tested |
| fig6-decoding-vv | tested |
| fig6-decoding-nvnv | tested |
| decoding-v-better-than-nv | tested |
| fig6-decoding-cross | tested |
| decoding-subepochs | tested |
| decoding-nonsignificant-cells | tested |
| eye-position-no-difference | blocked |
| aborted-trials-nv | blocked |
| longer-hold-no-difference | blocked |


In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import stats

import ppc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
CLAIMS = {c["id"]: c for c in yaml.safe_load(Path("../../claims/shi-2013.yaml").read_text())["claims"]}
def expected(cid):
    return [(e["quantity"], e["value"], e.get("tolerance")) for e in CLAIMS[cid].get("expected", [])]

## Loading

The data set is one 4.4 GB `data.tar.gz`. It is streamed once from the CRCNS bucket and each
session's MATLAB file is reduced to its trial table and spike times as it passes (`ppc.load_all`);
nothing is written to disk. The `.plx` file of the first session is parsed too, to check the
MATLAB extraction against Plexon's own file.

In [ ]:
t_start = time.time()
D = ppc.load_all()
print(f"streamed in {(time.time() - t_start) / 60:.1f} min")
import resource
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
import fsspec
with fsspec.open("https://crcnsarchive.s3-us-west-2.amazonaws.com/ppc-1/live/checksums.md5", "rt") as f:
    print(f.read())

## Data check

In [ ]:
mem = pd.DataFrame(D["members"], columns=["name", "size"])
mem["kind"] = mem.name.str.extract(r"(\.plxAnaAD\.dat|\.plxAnaAD\.mat|\.plxSpk\.dat|\.[A-Za-z0-9]+)$")[0]
print("members of data.tar.gz by kind:")
print(mem.kind.value_counts(dropna=False).to_string())
print("folders (recording days):", mem.name.str.extract(r"^data/(\d{6})/")[0].nunique())
print("session .mat files read:", len(D["sessions"]), " unreadable:", D["errors"])
names = pd.Series(sorted(D["sessions"]))
print("session file names: <mmddyy>_<n>", names.str.contains(r"/\d{6}_\d+\.mat$").sum(),
      " d<n>", names.str.contains(r"/d\d+\.mat$").sum(), " File<n>", names.str.contains(r"/File\d+\.mat$").sum())

The data descriptor lists `<session>.mat` with `strobed` (time stamps, event codes), `data`
(spike time stamps) and `waveforms`. It does not give units. The check below compares the first
session's `.mat` with its `.plx` file, read with Plexon's published layout.

In [ ]:
first = "data/010609/010609_1.mat"
s0 = D["sessions"][first]
plx = D["plx"]
st = s0["strobed_head"]
n = min(len(st), len(plx["strobed"]))
print("plx ADFrequency (Hz):", plx["adfreq"])
print("strobed events, .mat vs .plx: n", s0["n_strobes"], "; first", n, "codes identical:",
      bool(np.all(st[:n, 1] == plx["strobed"][:n, 1])),
      "; max time difference (s):", float(np.max(np.abs(st[:n, 0] - plx["strobed"][:n, 0]))))
u = s0["units"]["chan1a"]
for key, sp in plx["spikes"].items():
    m = min(len(sp), len(u))
    same = len(sp) == len(u) and np.allclose(sp, u, atol=1e-4)
    print(f"plx unit {key}: {len(sp)} spikes; equals .mat chan1a (ms / 1000): {same}")
print(".mat chan1a spikes span", round(u.min(), 2), "to", round(u.max(), 2), "s; strobes span",
      round(st[0, 0], 2), "to", round(s0["trials"].t0.max(), 2), "s")

### The trial record and the event codes

Each trial's events end with code 255 and an ASCII record (see `ppc.py`). `EventCodes.txt` in
the data set is a generic code list (names such as `TAPE = 1`, `DISPLAY = 10`) that does not
describe the codes used here, so their meaning is read from the timing below.

In [ ]:
allt = pd.concat([s["trials"].assign(session=k) for k, s in D["sessions"].items()], ignore_index=True)
print("trials:", len(allt), " completed (code 12):", int(allt.completed.sum()), " aborted (code 14):",
      int(allt.aborted.sum()))
print("record length (fields) per trial:")
print(allt.nrec.value_counts().to_string())
c = allt[allt.completed & allt.target.notna()]
print("completed trials by target code and field 14:")
print(pd.crosstab(c.target, c.cond).to_string())
iv = pd.DataFrame({
    "3 - t0 (start held, target on?)": c.t3 - c.t0,
    "8 - 3 (reaction time?)": c.t8 - c.t3,
    "target code - 8 (movement time?)": c.t_target - c.t8,
    "10 - target code": c.t10 - c.t_target,
    "12 - 10": c.t12 - c.t10,
    "13 - 12": c.t13 - c.t12,
})
print("intervals between events on completed trials (s):")
print(iv.describe(percentiles=[0.05, 0.5, 0.95]).T.round(3).to_string())

Reading of the codes, from the timing and the paper's paradigm (Methods, Experimental
paradigm). Code 3 comes 0.54 to 0.96 s after the trial starts (the start position is shown, then
held for 500 ms): it fits target onset, the go signal. Code 8 follows 0.22 to 0.58 s later (a
reaction time), and the target code 0.03 to 0.38 s after 8 (median 0.13 s): 8 fits the cursor
leaving the start window and the target code the cursor entering the target window. Code 10
follows the target code by 320 ms on every trial, the 300 ms after which the target is
extinguished (on a 32 ms strobe clock). From 10 to 12 is 0.83 to 1.86 s, the saccade to the
central fixation point and the 800 to 1,200 ms hold. Code 12 is taken as the end of the static
holding period, and 13, 32 ms later, as reward. These are readings of the data, not documented;
the clock check below tests the alignment.

In [ ]:
geo = []
for k, s in D["sessions"].items():
    t = s["trials"]
    if "target" not in t or t.target.isna().all():
        continue
    t = t[t.completed & t.target.isin(ppc.TARGETS)]
    row = t.target // 10
    col = t.target % 10
    geo.append(dict(session=k, n=len(t),
                    x_right_minus_left=t[col == 6].x.mean() - t[col == 4].x.mean(),
                    y_top_minus_bottom=t[row == 6].y.mean() - t[row == 4].y.mean(),
                    y_middle_row=t[row == 5].y.mean(), z=t.z.mean()))
geo = pd.DataFrame(geo)
print("target geometry per session, from the record's x, y, z (cm):")
print(geo.describe().round(2).to_string())
print("sessions with mean z > 12 cm:", (geo.z > 12).sum(), "; with y_top_minus_bottom > 9 cm:",
      (geo.y_top_minus_bottom > 9).sum())

The paper's targets are 5 cm apart horizontally and 4 cm vertically for monkey X (a rectangle),
5 cm along both axes for monkey B (a square), and Table 1 places monkey X's hand at depth z of
about 9.5 to 10.2 cm and monkey B's at 14.2 to 14.9 cm, with the middle row at y of about -4 cm
(X) and -0.8 cm (B). In every session with a record the targets are about 9.2 cm apart
horizontally and 7.4 cm vertically (twice 4.6 and 3.7 cm), the middle row is at y near -4 cm, and
depth is near 10 cm: the geometry of monkey X. No session has monkey B's geometry. The data set
descriptor says it holds 343 cells, 219 from X and 124 from B; the data are consistent with
monkey X only. A few sessions have z near 14 cm but monkey X's x and y layout.

In [ ]:
kept, sel = ppc.select_sessions(D["sessions"])
print(sel.status.value_counts().to_string())
print("sessions without a usable record:", sel[sel.status.str.startswith("no")].session.tolist())
ks = sel[sel.status.str.startswith("kept")]
print("kept sessions:", len(ks), "; sorted units in them:", int(ks.n_units.sum()),
      "; units per session:", ks.n_units.value_counts().sort_index().to_dict())
par = pd.Series([kept[k].params.iloc[0] for k in kept])
print("record fields 0 to 12 (task parameters) in kept sessions:")
print(par.value_counts().to_string())
early = [k for k in D["sessions"] if "/d" in k]
print("early d<n> files: record fields per trial",
      sorted({int(x) for k in early for x in D["sessions"][k]["trials"].nrec.unique()}))
for k in early:
    st_ = D["sessions"][k]["strobed_head"]
    c_ = st_[:, 1].astype(int)
    j = int(np.flatnonzero(c_ == 255)[0])
    nxt = np.flatnonzero(c_[j + 1:] == 0)
    rec = ppc.parse_record(c_[j + 1:j + 1 + (nxt[0] if len(nxt) else len(c_))])
    print(f"   {k}: {len(rec)} fields; fields 11 and 12 = {rec[11]:.0f}, {rec[12]:.0f}")

Kept are the sessions whose completed trials cover the 8 targets in both codes of field 14 with
at least five trials each, the paper's design ("Five trials were performed for each target",
Methods); where a cell has more, the first five in time are used. The early `d<n>` files have a
19 field record without positions, and field 11 (read as the minimum hold, 900 ms in most
sessions, with field 12, 300 ms, as its random part; see longer-hold-no-difference) is 500 or 600
ms there, below the paper's 800 ms hold; they are not used. Fields 1 and 7 vary between sessions (4
to 15) and may be window sizes; no file says.

### Which code is Vision

Field 14 splits each session's completed trials into two halves, five per target, as V and NV
should. No file says which is which. The conditions differ only from the end of the reach, when
the cursor is blanked on NV trials. The table counts aborted trials by whether they ended before
or after code 10.

In [ ]:
a = allt[allt.cond.isin([4, 7])].copy()
a["phase"] = np.where(a.completed, "completed", np.where(a.t10.notna(), "aborted after code 10", "aborted before code 10"))
tab = pd.crosstab(a.phase, a.cond)
print(tab.to_string())
pre = tab.loc["aborted before code 10"]; post = tab.loc["aborted after code 10"]
print("chi-square, aborts after code 10 against completed, by code:",
      stats.chi2_contingency(tab.loc[["aborted after code 10", "completed"]].values)[:2])
per = a[a.phase == "aborted after code 10"].groupby(["session", "cond"]).size().unstack(fill_value=0)
print("sessions with more aborts after code 10 on code 7 / on code 4 / equal:",
      int((per[7] > per[4]).sum()), int((per[4] > per[7]).sum()), int((per[4] == per[7]).sum()))
print("binomial test over sessions:", stats.binomtest(int((per[7] > per[4]).sum()),
      int((per[7] != per[4]).sum())).pvalue)

Aborts before code 10, when the two kinds of trial do not yet differ, are equal in number; aborts
after it are about 29% more frequent on code 7 trials. The paper reports more aborted trials
without vision of the cursor (Discussion, Additional factors influencing neural responses,
paragraph 1). Code 7 is therefore taken as No Vision and code 4 as Vision. This is an assumption
(`ppc.V_CODE`, `ppc.NV_CODE`). Outcomes that depend on the direction are marked as resting on it;
the other mapping is shown for the indexes and for the V against NV decoding, and for the
variances, FF and CV2 it would reverse the direction of the difference. Claims whose test is
symmetric in V and NV (the counts of ANOVA effects, the MANOVA and Levene tests, correlations)
do not depend on it. The neural results were not used to choose the mapping.

### The clock of spikes and events

If the spike times and the event times share a clock, the target should be decodable from spikes
after the reach and not before the target is shown. For each kept session's units, the fraction
with a significant target effect (one-way ANOVA on counts over the 80 trials, P < 0.05) is
computed in 800 ms windows ending at code 12, and in windows shifted in time.

In [ ]:
def units_of(kept, all_units=False):
    out = []
    for k, t in kept.items():
        us = D["sessions"][k]["units"]
        names_ = list(us)[:None if all_units else 1]
        for u_ in names_:
            out.append((k, u_, us[u_]))
    return out
UNITS_ALL = units_of(kept, all_units=True)
shifts = [-6.0, -4.0, -3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 3.0]
frac = []
for sh in shifts:
    sig = []
    for k, u_, sp in UNITS_ALL:
        t = kept[k]
        cnt = ppc.unit_counts(t, sp, (sh - 0.8, sh))
        groups = [cnt[t.target.values == tg] for tg in ppc.TARGETS]
        if np.all(cnt == cnt[0]):
            sig.append(False)
            continue
        sig.append(stats.f_oneway(*groups).pvalue < 0.05)
    frac.append(np.mean(sig))
print(pd.DataFrame({"window end relative to code 12 (s)": shifts, "fraction of units with a target effect": np.round(frac, 3)}).to_string(index=False))
pre_t = []
for k, u_, sp in UNITS_ALL:
    t = kept[k]
    cnt = ppc.unit_counts(t, sp, (-0.8, 0.0), align="t0")
    groups = [cnt[t.target.values == tg] for tg in ppc.TARGETS]
    pre_t.append(False if np.all(cnt == cnt[0]) else stats.f_oneway(*groups).pvalue < 0.05)
print("window of 800 ms before the trial's first event:", round(float(np.mean(pre_t)), 3), "(chance 0.05)")
print("units:", len(UNITS_ALL))

The target is decoded in about a third or more of the units in the windows ending at or before
code 12 back to the reach, and in about 5% (chance) before the trial starts, when the target is
not yet known. Spike times and event times share a clock in the data as a whole; this check does
not establish it for each session separately.

### Per-trial measures

For each kept unit and trial: spike counts in the last 800 ms of the hold, its first and last
400 ms, and the 500 ms centred on code 8; the smoothed rate of the paper's PSTH averaged over the
hold window; and the spike times in the hold window (for CV2). Two populations are used. The
paper has one neuron per session (Table 1 counts sessions; the Results give "N = 343 sessions"
and 343 neurons), so the main population takes the first sorted unit of each kept session; every
sorted unit is shown as a variant. Units without a spike in any hold window are left out.

In [ ]:
def measures(unit_list, v_code=ppc.V_CODE, nv_code=ppc.NV_CODE):
    rows = []
    for k, u_, sp in unit_list:
        t = kept[k]
        r = dict(session=k, unit=u_)
        for w, win in ppc.WINDOWS.items():
            cnt = ppc.unit_counts(t, sp, win)
            r[w] = ppc.cell_array(t, cnt / (win[1] - win[0]), v_code, nv_code)
            r[w + "_count"] = ppc.cell_array(t, cnt, v_code, nv_code)
        mv = ppc.unit_counts(t, sp, (-ppc.MOVE_HALF, ppc.MOVE_HALF), align="t8")
        r["move"] = ppc.cell_array(t, mv / (2 * ppc.MOVE_HALF), v_code, nv_code)
        r["psth"] = ppc.cell_array(t, ppc.psth_rate(t, sp, ppc.WINDOWS["hold"]), v_code, nv_code)
        hold_sp = np.empty(len(t), dtype=object)
        for i, t12 in enumerate(t.t12.values):
            hold_sp[i] = ppc.spikes_in(sp, t12 - 0.8, t12)
        arr = np.empty((2, 8, ppc.N_PER_CELL), dtype=object)
        for ci, code in enumerate((v_code, nv_code)):
            for ti, tg in enumerate(ppc.TARGETS):
                idx = np.flatnonzero((t.cond.values == code) & (t.target.values == tg))
                for j, i in enumerate(idx[:ppc.N_PER_CELL]):
                    arr[ci, ti, j] = hold_sp[i]
        r["hold_spikes"] = arr
        rows.append(r)
    M = pd.DataFrame(rows)
    active = M.hold_count.apply(lambda a: a.sum() > 0)
    return M[active].reset_index(drop=True), M[~active][["session", "unit"]]

def add_anova(M, key="hold", prefix=""):
    p = np.array([ppc.anova2(y) for y in M[key]])
    M[prefix + "p_pos"], M[prefix + "p_vis"], M[prefix + "p_int"] = p[:, 0], p[:, 1], p[:, 2]
    return M

M1, drop1 = measures(units_of(kept))
MA, dropA = measures(UNITS_ALL)
for M in (M1, MA):
    add_anova(M)
print("main population (first unit per session):", len(M1), "units; left out, no hold spikes:", len(drop1))
print("all sorted units:", len(MA), "units; left out, no hold spikes:", len(dropA))
r = np.stack(M1.hold.values)
print("hold rate (spikes/s) of main population: range", r.min(), "to", r.max(), "; median of unit means",
      round(float(np.median(r.mean(axis=(1, 2, 3)))), 2))
print("ANOVA left undefined (zero error variance):", int(M1.p_pos.isna().sum()), "main;", int(MA.p_pos.isna().sum()), "all")

## behavior-mean-position-sessions

> behavior-mean-position-sessions: Few sessions showed a significant difference in mean arm endpoint position between V and NV at a target; the average percentage across animals and targets was 6.9 +/- 1.5% (one-factor MANOVA, P < 0.05).

In [ ]:
beh = []
for k, t in kept.items():
    for tg in ppc.TARGETS:
        xv = t[(t.target == tg) & (t.cond == ppc.V_CODE)][["x", "y", "z"]].values
        xn = t[(t.target == tg) & (t.cond == ppc.NV_CODE)][["x", "y", "z"]].values
        F, p = ppc.hotelling2(xv, xn)
        row = dict(session=k, target=ppc.TARGET_DEG[tg], p_manova=p)
        for ax, i in zip("xyz", range(3)):
            row["p_levene_mean_" + ax] = stats.levene(xv[:, i], xn[:, i], center="mean").pvalue
            row["p_levene_median_" + ax] = stats.levene(xv[:, i], xn[:, i], center="median").pvalue
        beh.append(row)
beh = pd.DataFrame(beh)
print("session x target pairs:", len(beh), "; MANOVA undefined:", int(beh.p_manova.isna().sum()))
pt = beh.groupby("target").p_manova.apply(lambda p: 100 * np.mean(p < 0.05))
print("percentage of sessions with a significant V against NV difference, by target (monkey X):")
print(pt.round(1).to_string())
print(f"(a) mean of the 8 target percentages: {pt.mean():.2f} +/- {pt.std(ddof=1):.2f} (SD)")
print(f"(b) pooled over session and target pairs: {100 * np.mean(beh.p_manova < 0.05):.2f}")
print("expected:", expected("behavior-mean-position-sessions"))

**Result: not reproduced.** In monkey X, 16.1% of session and target pairs (2,872 pairs, 359
sessions) show a significant V against NV difference in endpoint position by the two-group MANOVA,
both as the mean of the 8 target percentages (16.12 +/- 4.27) and pooled (16.12), against 6.9 +/-
1.5%. Each trial has one endpoint in the data (the record's x, y, z), not the paper's mean over the
800 ms hold; monkey B is not in the data set.

## behavior-variability-sessions

> behavior-variability-sessions: Few sessions showed a significant difference in endpoint variability between V and NV; along the horizontal, vertical and depth axes the average percentages were 5 +/- 1.9%, 4.7 +/- 1.6% and 6 +/- 0.9% (Levene's test, P < 0.05).

In [ ]:
rows = []
for centre in ("mean", "median"):
    for ax in "xyz":
        col = f"p_levene_{centre}_{ax}"
        pt_ = beh.groupby("target")[col].apply(lambda p: 100 * np.mean(p < 0.05))
        rows.append(dict(centre=centre, axis=ax, mean_of_targets=pt_.mean(), sd_of_targets=pt_.std(ddof=1),
                         pooled=100 * np.mean(beh[col] < 0.05)))
lev = pd.DataFrame(rows)
print(lev.round(2).to_string(index=False))
exp_ = {e[0].split(", ")[-1].split(" ")[0]: e for e in expected("behavior-variability-sessions")}
for centre in ("mean", "median"):
    for avg in ("mean_of_targets", "pooled"):
        ok = []
        for ax, (q, v, tol) in zip("xyz", expected("behavior-variability-sessions")):
            m = lev[(lev.centre == centre) & (lev.axis == ax)][avg].item()
            ok.append(abs(m - v) <= tol)
        print(f"centre {centre}, {avg}: within tolerance x, y, z = {ok}")

**Result: not reproduced.** With Levene's test centred on the mean, 8.1%, 8.1% and 7.0% of pairs
differ in x, y and z (paper 5 +/- 1.9, 4.7 +/- 1.6, 6 +/- 0.9): z is within tolerance, x and y are
not. Centred on the median (Brown-Forsythe), 0.4 to 0.7%, none within. No reading puts all three
axes within tolerance.

## table1-endpoint-means and table1-endpoint-sd

> table1-endpoint-means: Mean endpoint positions per target in the V and NV conditions are those of Table 1, and differ between conditions by at most 0.11 cm at every target, coordinate and animal.
>
> table1-endpoint-sd: The across-session SDs of endpoint position per target are those of Table 1, about 0.5 to 1.3 cm for monkey X and 0.4 to 0.6 cm for monkey B.

Only monkey X's 48 values can be compared; the data set holds no monkey B session.

In [ ]:
sm_ = pd.concat([t.assign(session=k) for k, t in kept.items()]).groupby(["session", "target", "cond"])[["x", "y", "z"]].mean().reset_index()
print("sessions:", sm_.session.nunique())
rows = []
exp_m = expected("table1-endpoint-means")[:48]
exp_s = expected("table1-endpoint-sd")[:48]
i = 0
for tg in ppc.TARGETS:
    for ax in "xyz":
        for code, lab in ((ppc.NV_CODE, "NV"), (ppc.V_CODE, "V")):
            vals = sm_[(sm_.target == tg) & (sm_.cond == code)][ax]
            q, v, tol = exp_m[i]
            qs, vs, tols = exp_s[i]
            assert f"{ppc.TARGET_DEG[tg]} degrees, {ax}, {lab}" in q
            rows.append(dict(target=ppc.TARGET_DEG[tg], axis=ax, cond=lab, mean=vals.mean(), paper_mean=v, tol=tol,
                             sd=vals.std(ddof=1), paper_sd=vs, tol_sd=tols))
            i += 1
T1 = pd.DataFrame(rows)
T1["in_tol"] = (T1["mean"] - T1.paper_mean).abs() <= T1.tol
off = T1.assign(d=T1["mean"] - T1.paper_mean).groupby("axis").d.mean()
T1["mean_offset_removed"] = T1["mean"] - T1.axis.map(off)
T1["in_tol_offset_removed"] = (T1.mean_offset_removed - T1.paper_mean).abs() <= T1.tol
print(T1[["target", "axis", "cond", "mean", "paper_mean", "tol", "in_tol", "mean_offset_removed", "in_tol_offset_removed"]].round(2).to_string(index=False))
print("mean offset per axis (data minus Table 1, cm):", off.round(3).to_dict())
print(f"within tolerance: {T1.in_tol.mean() * 100:.1f}% of 48; after removing the offset: {T1.in_tol_offset_removed.mean() * 100:.1f}%")
dv = T1.pivot_table(index=["target", "axis"], columns="cond", values="mean")
d = (dv["V"] - dv["NV"]).abs()
print(f"largest |V - NV| difference of the means: {d.max():.3f} cm (bound 0.11 + 10% = 0.121); at {d.idxmax()}")
dvf = T1.pivot_table(index=["target", "axis"], columns="cond", values="mean")
print("under the other code mapping the |V - NV| differences are the same (absolute values).")

**Result: partially reproduced.** All 48 of monkey X's means are within the Table 1 SDs (100%,
also after removing the mean offsets of -0.10, -0.17 and +0.26 cm), but the largest V minus NV
difference of the means is 0.150 cm (135 degrees, y), above the 0.121 cm bound. Monkey B's 48
values cannot be tested; the data set holds 359 usable monkey X sessions against the paper's 219.

> table1-endpoint-sd: The across-session SDs of endpoint position per target are those of Table 1, about 0.5 to 1.3 cm for monkey X and 0.4 to 0.6 cm for monkey B.

In [ ]:
T1["sd_in_tol"] = (T1.sd - T1.paper_sd).abs() <= T1.tol_sd
print(T1[["target", "axis", "cond", "sd", "paper_sd", "tol_sd", "sd_in_tol"]].round(3).to_string(index=False))
tr_sd = pd.concat([t.assign(session=k) for k, t in kept.items()]).groupby(["target", "cond"])[["x", "y", "z"]].std()
T1["sd_trials"] = [tr_sd.loc[(tg, {"V": ppc.V_CODE, "NV": ppc.NV_CODE}[c_]), ax] for tg, c_, ax in
                   zip([{v: k for k, v in ppc.TARGET_DEG.items()}[d_] for d_ in T1.target], T1.cond, T1.axis)]
print(f"variant, SD over all trials pooled across sessions: {T1.sd_trials.min():.2f} to {T1.sd_trials.max():.2f} cm; "
      f"within 10% of Table 1: {100 * np.mean((T1.sd_trials - T1.paper_sd).abs() <= T1.tol_sd):.1f}%")
print(f"SDs within 10%: {T1.sd_in_tol.mean() * 100:.1f}% of 48; range of SDs {T1.sd.min():.2f} to {T1.sd.max():.2f} cm "
      f"(Table 1, monkey X: {T1.paper_sd.min():.2f} to {T1.paper_sd.max():.2f}); median ratio data / paper {np.median(T1.sd / T1.paper_sd):.2f}")

**Result: not reproduced.** 8.3% of monkey X's 48 SDs over sessions are within 10% of Table 1; the
SDs run from 0.32 to 0.67 cm against 0.50 to 1.28 cm, a median of 0.55 times the paper's. The SD
over all trials (a variant) runs from 0.41 to 0.73 cm, 14.6% within.

## fig3-example-neuron

> fig3-example-neuron: An example neuron fired most when the hand was held at the lower left target (225 degrees) in both V and NV; its hold-period rate showed a main effect of position (P < 0.05) but no effect of vision (P = 0.52) and no interaction (P = 0.64).

In [ ]:
def example_search(M):
    rows = []
    for _, r in M.iterrows():
        m = r.hold.mean(axis=2)
        rows.append(dict(session=r.session, unit=r.unit, peak_V=ppc.TARGETS[int(np.argmax(m[0]))],
                         peak_NV=ppc.TARGETS[int(np.argmax(m[1]))], p_pos=r.p_pos, p_vis=r.p_vis, p_int=r.p_int))
    E = pd.DataFrame(rows)
    E["peak225_both"] = (E.peak_V == 44) & (E.peak_NV == 44)
    E["dist"] = np.hypot((E.p_vis - 0.52) / 0.052, (E.p_int - 0.64) / 0.064)
    E["match"] = E.peak225_both & (E.p_pos < 0.05) & ((E.p_vis - 0.52).abs() <= 0.052) & ((E.p_int - 0.64).abs() <= 0.064)
    return E
for lab, M in (("main", M1), ("all units", MA)):
    E = example_search(M)
    print(f"{lab}: units with the peak at 225 deg in V and NV and a position effect:",
          int((E.peak225_both & (E.p_pos < 0.05)).sum()), "; candidates matching all:", int(E.match.sum()))
    print(E[E.match].round(3).to_string(index=False) if E.match.any() else
          E[E.peak225_both & (E.p_pos < 0.05)].sort_values("dist").head(3).round(3).to_string(index=False))

**Result: not reproduced.** Of the 359 units, 8 peak at 225 degrees in both conditions with a
position effect (10 of 483 among all units), and none has vision and interaction P values within
0.52 +/- 0.052 and 0.64 +/- 0.064; the closest (073009_4, chan1a) has P = 0.364 and 0.537. The
paper does not identify the neuron, which may be from monkey B, absent here.

## movement-epoch-tuning

> movement-epoch-tuning: In a 500 ms epoch centred on movement onset, 41% (139 of 343) of neurons were modulated by the movement vector (main and/or interaction effects, ANOVA, P < 0.05).

Movement onset is read as code 8 (reading (a) of the criterion, an event code; see the data
check). No continuous arm position is in the data set, so reading (b) cannot be run: the record
gives one position per trial.

In [ ]:
for lab, M in (("main", M1), ("all units", MA)):
    p = np.array([ppc.anova2(y) for y in M.move])
    sig = (p[:, 0] < 0.05) | (p[:, 2] < 0.05)
    n = len(M)
    print(f"{lab}: {sig.sum()} of {n} units ({100 * sig.mean():.1f}%) with a target main effect or interaction; "
          f"scaled to 343: {343 * sig.mean():.0f} (paper 139, 41%)")
# sensitivity: window centred on the target-code event (end of the reach) and on code 3
for lab, al in (("centred on target acquired", "t_target"), ("centred on code 3", "t3")):
    sig = []
    for k, u_, sp in units_of(kept):
        t = kept[k]
        cnt = ppc.unit_counts(t, sp, (-0.25, 0.25), align=al)
        y = ppc.cell_array(t, cnt / 0.5)
        if y.sum() == 0:
            continue
        p = ppc.anova2(y)
        sig.append((p[0] < 0.05) | (p[2] < 0.05))
    print(f"variant, 500 ms {lab}: {100 * np.mean(sig):.1f}% of {len(sig)}")

**Result: not reproduced.** With movement onset read as code 8, 187 of 359 units (52.1%; 179 when
scaled to 343) have a target main effect or interaction in the 500 ms around it, against 139 +/- 13.9
(41%); all units give 49.9% (171). The result rests on reading code 8 as movement onset; a window
centred on the target code event gives 65.5%, one on code 3 6.1%.

## hold-position-main-effect, hold-vision-main-effect, hold-interaction

> hold-position-main-effect: In the 800 ms static holding period, 39% (135 of 343) of neurons showed a significant main effect of position.
>
> hold-vision-main-effect: Only 13% (44 of 343) of neurons showed a main effect of the visual conditions in the holding period.
>
> hold-interaction: Interaction effects of position and vision were found in 9% (27 of 343) of neurons.

In [ ]:
def effect_table(M, prefix=""):
    pp, pv, pi = M[prefix + "p_pos"], M[prefix + "p_vis"], M[prefix + "p_int"]
    n = len(M)
    return dict(n=n, position=int((pp < 0.05).sum()), vision=int((pv < 0.05).sum()), interaction=int((pi < 0.05).sum()))
for lab, M, key in (("main, spike count rate", M1, "hold"), ("main, smoothed PSTH rate", M1, "psth"),
                    ("all units, spike count rate", MA, "hold")):
    Mx = add_anova(M.copy(), key, "x_")
    e = effect_table(Mx, "x_")
    n = e["n"]
    print(f"{lab}: n = {n}; position {e['position']} ({100 * e['position'] / n:.1f}%), vision {e['vision']} "
          f"({100 * e['vision'] / n:.1f}%), interaction {e['interaction']} ({100 * e['interaction'] / n:.1f}%); "
          f"scaled to 343: {343 * e['position'] / n:.0f}, {343 * e['vision'] / n:.0f}, {343 * e['interaction'] / n:.0f}")
print("paper: position 135/343 (39.4%), vision 44/343 (12.8%), interaction 27/343 (7.9%) or 9% (31)")
print("false positive rate of the ANOVA under shuffled labels (main population, 1 shuffle per unit):")
rng = np.random.default_rng(1)
fp = []
for y in M1.hold:
    flat = y.reshape(-1).copy()
    rng.shuffle(flat)
    fp.append(ppc.anova2(flat.reshape(y.shape)))
fp = np.array(fp)
print("  position, vision, interaction:", np.round(np.nanmean(fp < 0.05, axis=0) * 100, 1), "%")

**Result: reproduced.** 141 of 359 units (39.3%; 135 scaled to 343) have a position main effect
in the last 800 ms of the hold, against 135 of 343 (39.4%). The smoothed PSTH rate gives 142 (39.6%)
and all 483 units 36.4% (125 scaled), both within tolerance. Shuffled labels give 4.2%.

**Result: reproduced.** 49 of 359 units (13.6%; 47 scaled to 343) have a vision main effect,
against 44 +/- 4.4, and fewer than the 141 with a position effect; all units give 14.3% (49 scaled).
The count does not depend on which code is V.

**Result: reproduced.** 27 of 359 units (7.5%; 26 scaled to 343) have an interaction, within
reading (a), 27 +/- 2.7, and not reading (b), 31 +/- 3.1. All units give 7.2% (25 scaled).

## hold-subepochs

> hold-subepochs: Similar percentages were found when the first 400 ms and the last 400 ms of the hold were analyzed separately, 31%, 9% and 5% and 29%, 10% and 4% for position main effect, vision main effect and interaction.

In [ ]:
exp_sub = [e[1:] for e in expected("hold-subepochs")]
for lab, M in (("main", M1), ("all units", MA)):
    vals = []
    for w in ("first400", "last400"):
        Mx = add_anova(M.copy(), w, "x_")
        e = effect_table(Mx, "x_")
        vals += [100 * e["position"] / e["n"], 100 * e["vision"] / e["n"], 100 * e["interaction"] / e["n"]]
    ok = [abs(v - ev) <= tol for v, (ev, tol) in zip(vals, exp_sub)]
    print(f"{lab}: first 400 ms {np.round(vals[:3], 1)}%, last 400 ms {np.round(vals[3:], 1)}%; within tolerance: {sum(ok)} of 6 {ok}")
print("paper: 31/9/5% and 29/10/4%")

**Result: not reproduced.** In the first and last 400 ms, 35.7/11.4/7.0% and 28.7/13.6/5.0% of
the 359 units have position, vision and interaction effects, against 31/9/5% and 29/10/4%: 1 of 6
within tolerance. All 483 units give 3 of 6 (31.5/11.2/6.2% and 27.1/13.3/4.3%). The position
percentages are close; the vision and interaction ones are higher than the paper's.

## hold-overall-effects

> hold-overall-effects: Over the full hold, 44% of neurons showed some effect of position (main and/or interaction), 19% some effect of vision, and 11% effects of both position and vision.

In [ ]:
def groups(M):
    pp, pv, pi = M.p_pos < 0.05, M.p_vis < 0.05, M.p_int < 0.05
    return dict(any_position=pp | pi, any_vision=pv | pi, both=(pp & pv) | pi,
                position_only=pp & ~pv & ~pi, position_vision=(pp & pv) | pi, vision_only=pv & ~pp & ~pi,
                none=~pp & ~pv & ~pi)
for lab, M in (("main", M1), ("all units", MA)):
    g = groups(M)
    print(f"{lab} (n = {len(M)}): any position {100 * g['any_position'].mean():.1f}%, any vision {100 * g['any_vision'].mean():.1f}%, "
          f"both {100 * g['both'].mean():.1f}%")
print("paper: 44%, 19%, 11%")

**Result: partially reproduced.** 40.4% of units show some position effect (44 +/- 4.4) and 18.7%
some vision effect (19 +/- 1.9), both within tolerance; 13.1% show both, above 11 +/- 1.1. All units
give 38.3%, 18.8% and 12.2%.

## correlation-endpoint-datasets

> correlation-endpoint-datasets: Correlations between V and NV responses were the same where mean endpoint positions differed between conditions (r = 0.9301) and where they did not (r = 0.9281), so firing rate differences did not reflect endpoint differences.

In [ ]:
sigpair = set(map(tuple, beh[beh.p_manova < 0.05][["session", "target"]].values))
def corr_sets(M, trial_level=False):
    d1, d2 = [], []
    for _, r in M.iterrows():
        for ti, tg in enumerate(ppc.TARGETS):
            if trial_level:
                pairs = list(zip(r.hold[0, ti], r.hold[1, ti]))
            else:
                pairs = [(r.hold[0, ti].mean(), r.hold[1, ti].mean())]
            (d1 if (r.session, ppc.TARGET_DEG[tg]) in sigpair else d2).extend(pairs)
    return np.array(d1), np.array(d2)
def perm_r(d1, d2, n_perm=1000, seed=0):
    rng = np.random.default_rng(seed)
    obs = stats.spearmanr(d1[:, 0], d1[:, 1])[0] - stats.spearmanr(d2[:, 0], d2[:, 1])[0]
    pool = np.vstack([d1, d2])
    diffs = np.empty(n_perm)
    for i in range(n_perm):
        idx = rng.permutation(len(pool))
        a_, b_ = pool[idx[:len(d1)]], pool[idx[len(d1):]]
        diffs[i] = stats.spearmanr(a_[:, 0], a_[:, 1])[0] - stats.spearmanr(b_[:, 0], b_[:, 1])[0]
    return obs, np.mean(np.abs(diffs) >= abs(obs))
for lab, M in (("main", M1), ("all units", MA)):
    for tl in (False, True):
        d1, d2 = corr_sets(M, tl)
        r1, r2 = stats.spearmanr(d1[:, 0], d1[:, 1])[0], stats.spearmanr(d2[:, 0], d2[:, 1])[0]
        obs, p = perm_r(d1, d2)
        print(f"{lab}, reading {'(b) trials' if tl else '(a) means'}: data set 1 r = {r1:.4f} (n = {len(d1)}), "
              f"data set 2 r = {r2:.4f} (n = {len(d2)}); difference {obs:.4f}, permutation P = {p:.3f}")
d1, d2 = corr_sets(M1)
allr = stats.spearmanr(np.vstack([d1, d2])[:, 0], np.vstack([d1, d2])[:, 1])[0]
print(f"all responses (main, means): r = {allr:.4f}")

**Result: reproduced (resting on an assumption).** With each cell's V and NV mean rates at a
target as the pair (reading (a)), r is 0.9237 where endpoints differed (463 pairs) and 0.9154 where
they did not (2,409), both within 0.093 of the paper's, and the permutation test finds no difference
(P = 0.469). With single trials as pairs (reading (b)), r is 0.794 and 0.767, outside tolerance, and
the difference is significant (P = 0.025). Reading (a) is adopted, as its r values are of the
paper's size; all units give the same pattern (P = 0.267 and 0.001).

## fig4-indx1-negative and fig4-indx1-means

> fig4-indx1-negative: For most cells INDX1 was negative (activity lower with vision than without), with 84%, 77%, 85% and 84% negative among all cells, Position Only, Position/Vision and Vision Only cells (338, 110, 38 and 27 cells after outlier removal).
>
> fig4-indx1-means: Mean INDX1 was least negative for Position Only cells (-13%) and most negative for Position/Vision cells (-31%).

In [ ]:
def indexes(M):
    out = []
    for _, r in M.iterrows():
        m = r.hold.mean(axis=2)
        pnv, pv = int(np.argmax(m[1])), int(np.argmax(m[0]))
        nv = m[1, pnv]
        out.append(dict(pnv=pnv, pv=pv,
                        indx1=np.nan if nv == 0 else (m[0, pnv] - nv) / nv * 100,
                        indx2=np.nan if nv == 0 else (m[0, pv] - nv) / nv * 100))
    return pd.DataFrame(out, index=M.index)
def fences(x):
    q1, q3 = np.nanpercentile(x, [25, 75])
    return q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1), q3 - q1
def outliers(M, col, reading):
    x = M[col]
    g = groups(M)
    if reading == "a":
        lo, hi, _ = fences(x)
        return (x < lo) | (x > hi)
    if reading == "b":
        o = pd.Series(False, index=M.index)
        for gname in ("position_only", "position_vision", "vision_only", "none"):
            sub = x[g[gname]]
            lo, hi, _ = fences(sub)
            o[g[gname]] = (sub < lo) | (sub > hi)
        return o
    if reading == "c":
        _, _, iqr = fences(x)
        return x > 1.5 * iqr
def fig4(M, col="indx1", reading="a", outlier_col=None):
    I = indexes(M)
    M = M.join(I)
    o = outliers(M, outlier_col or col, reading) | M[col].isna()
    g = groups(M)
    res = {}
    for gname, mask in (("all", pd.Series(True, index=M.index)), ("position_only", g["position_only"]),
                        ("position_vision", g["position_vision"]), ("vision_only", g["vision_only"])):
        x = M[col][mask & ~o]
        res[gname] = dict(n=len(x), neg=100 * np.mean(x < 0), mean=x.mean(), removed=int((mask & o).sum()))
    return pd.DataFrame(res).T, M, o
for lab, M in (("main", M1), ("all units", MA)):
    for reading in ("a", "b", "c"):
        R, _, o = fig4(M, "indx1", reading)
        print(f"{lab}, outlier reading ({reading}): removed {int(o.sum())}")
        print(R.round(1).to_string())
print("paper: negative 84/77/85/84%; n 338/110/38/27; mean INDX1 Position Only -13%, Position/Vision -31%")
print("under the other code mapping (V = 7), main, reading (a):")
M1f = M1.copy()
M1f["hold"] = [y[::-1] for y in M1.hold]
R, _, _ = fig4(M1f, "indx1", "a")
print(R.round(1).to_string())

**Result: partially reproduced.** After removing outliers (reading (a), 3 cells), INDX1 is
negative in 77.8% of all cells, 68.4% of Position Only, 75.0% of Position/Vision and 75.0% of
Vision Only cells (paper 84, 77, 85, 84%): only the first is within tolerance, and the others are
0.6 to 1.5 points below it. Readings (b) and (c) give the same. Group sizes are 356, 98, 44 and 20
(paper 338, 110, 38, 27). Among all 483 units the four percentages (81.4, 71.4, 76.8, 81.2%) are
within tolerance. INDX1 is mostly negative also under the other code mapping (80.5% of all cells).

> fig4-indx1-means: Mean INDX1 was least negative for Position Only cells (-13%) and most negative for Position/Vision cells (-31%).

**Result: partially reproduced.** Mean INDX1 is -12.0% for Position Only cells (paper -13 +/-
1.3, within) and -24.2% for Position/Vision cells (paper -31 +/- 3.1, not within), and the most
negative of the four populations is all cells (-24.9%), not Position/Vision. Position Only is the
least negative, as in the paper. Outlier readings (b) and (c) and all units give the same pattern.

## indx1-tuning-shift

> indx1-tuning-shift: Among Position/Vision cells, those with their maximal response at the same or an adjacent location in the two conditions (22 of 38, 58%) had a mean INDX1 of -24%, and the others a more negative mean of -40%.

In [ ]:
grid = {56: (1, 0), 66: (1, 1), 65: (0, 1), 64: (-1, 1), 54: (-1, 0), 44: (-1, -1), 45: (0, -1), 46: (1, -1)}
for lab, M in (("main", M1), ("all units", MA)):
    for reading in ("a", "b", "c"):
        _, Mi, o = fig4(M, "indx1", reading)
        pvc = groups(Mi)["position_vision"] & ~o
        sub = Mi[pvc]
        ring = ((sub.pv - sub.pnv) % 8).isin([0, 1, 7])
        gpos = lambda i: np.array(grid[ppc.TARGETS[i]])
        gridadj = np.array([np.max(np.abs(gpos(a_) - gpos(b_))) <= 1 for a_, b_ in zip(sub.pv, sub.pnv)])
        for adj_lab, adj in (("ring", ring.values), ("grid", gridadj)):
            print(f"{lab}, outliers ({reading}), adjacency {adj_lab}: {adj.sum()} of {len(sub)} same or adjacent, "
                  f"mean INDX1 {sub.indx1[adj].mean():.1f}% vs others {sub.indx1[~adj].mean():.1f}% (n {int((~adj).sum())})")
print("paper: 22 of 38, -24% vs -40%")

**Result: not reproduced.** Of 44 Position/Vision cells, 33 (ring) or 34 (grid) have their V and
NV peaks at the same or adjacent targets (paper 22 of 38), with mean INDX1 -23.0% (paper -24, within)
against -27.6% for the others (paper -40, not within).

## indx2-means

> indx2-means: With an index based on each condition's own preferred location (INDX2), rate differences were weaker but still biased toward suppression, with means of -9%, -13% and -13% for Position Only, Vision Only and Position/Vision cells.

In [ ]:
for lab, M in (("main", M1), ("all units", MA), ("main, other mapping", M1f)):
    for oc in ("indx1", "indx2"):
        for reading in ("a", "b", "c"):
            R, _, o = fig4(M, "indx2", reading, outlier_col=oc)
            print(f"{lab}, outliers on {oc} ({reading}): mean INDX2 Position Only {R.loc['position_only', 'mean']:.1f}, "
                  f"Vision Only {R.loc['vision_only', 'mean']:.1f}, Position/Vision {R.loc['position_vision', 'mean']:.1f} "
                  f"(n {int(R.loc['position_only', 'n'])}, {int(R.loc['vision_only', 'n'])}, {int(R.loc['position_vision', 'n'])})")
print("paper: -9, -13, -13")

**Result: not reproduced.** Mean INDX2 is -0.5%, +12.7% and +7.5% for Position Only, Vision Only
and Position/Vision cells (outliers on INDX1, reading (a)), against -9, -13 and -13%. No reading,
population or code mapping gives three negative means within tolerance; the closest, outliers on
INDX2 by reading (c), gives -1.7, -10.5 and -9.5%.

## rate-variance-nv-higher

> rate-variance-nv-higher: Spike rate variance at the NV preferred location tended to be higher in NV than in V, on average 56 against 30 (spikes/s)^2 across cells.

In [ ]:
_, Mi1, o1 = fig4(M1, "indx1", "a")
for lab, M, keep in (("main, all cells", M1, None), ("main, after Fig. 4 outliers (a)", M1, ~o1), ("all units", MA, None)):
    I = indexes(M)
    for ddof in (1, 0):
        vv = np.array([r.hold[0, p].var(ddof=ddof) for r, p in zip(M.itertuples(), I.pnv)])
        vn = np.array([r.hold[1, p].var(ddof=ddof) for r, p in zip(M.itertuples(), I.pnv)])
        if keep is not None:
            vv, vn = vv[keep.values], vn[keep.values]
        print(f"{lab}, ddof {ddof}: mean variance NV {vn.mean():.1f}, V {vv.mean():.1f} (spikes/s)^2; n {len(vv)}; "
              f"NV > V in {100 * np.mean(vn > vv):.0f}% of cells; median NV {np.median(vn):.1f}, V {np.median(vv):.1f}")
print("paper: NV 56, V 30")

**Result: partially reproduced (resting on an assumption).** At the NV preferred location the mean
rate variance across the 359 cells is 46.4 (spikes/s)^2 in NV and 23.1 in V, higher in NV in 67% of
cells, as the paper's 56 against 30, but both below tolerance (50.4 and 27). After the Fig. 4
outliers it is 46.7 and 23.0; all units give 48.5 and 25.6. Which condition is NV rests on the
code mapping.

## fig5-fano-factor

> fig5-fano-factor: The Fano factor was significantly larger in NV (1.54) than in V (1.13) (permutation test, P < 0.05).

In [ ]:
def ff_slope(m, v):
    return np.sum(m * v) / np.sum(m * m)
def ff_test(M, keep=None, seed=2, n_perm=1000):
    I = indexes(M)
    cm = np.array([[r.hold_count[c_, p].mean() for c_ in (0, 1)] for r, p in zip(M.itertuples(), I.pnv)])
    cv = np.array([[r.hold_count[c_, p].var(ddof=1) for c_ in (0, 1)] for r, p in zip(M.itertuples(), I.pnv)])
    if keep is not None:
        cm, cv = cm[keep], cv[keep]
    ffv, ffn = ff_slope(cm[:, 0], cv[:, 0]), ff_slope(cm[:, 1], cv[:, 1])
    rng = np.random.default_rng(seed)
    pm, pv_ = np.concatenate([cm[:, 0], cm[:, 1]]), np.concatenate([cv[:, 0], cv[:, 1]])
    n = len(cm)
    d = np.empty(n_perm)
    for i in range(n_perm):
        idx = rng.permutation(2 * n)
        d[i] = ff_slope(pm[idx[n:]], pv_[idx[n:]]) - ff_slope(pm[idx[:n]], pv_[idx[:n]])
    obs = ffn - ffv
    return ffv, ffn, np.mean(np.abs(d) >= abs(obs)), np.mean(d >= obs), n, cm, cv
I1 = indexes(M1)
cm_all = np.array([[r.hold_count[c_, p].mean() for c_ in (0, 1)] for r, p in zip(M1.itertuples(), I1.pnv)])
cv_all = np.array([[r.hold_count[c_, p].var(ddof=1) for c_ in (0, 1)] for r, p in zip(M1.itertuples(), I1.pnv)])
tuk = np.zeros(len(M1), bool)
for arr in (cm_all[:, 0], cm_all[:, 1], cv_all[:, 0], cv_all[:, 1]):
    lo, hi, _ = fences(arr)
    tuk |= (arr < lo) | (arr > hi)
for lab, M, keep in (("main, Fig. 4 outliers (a) removed", M1, (~o1).values), ("main, Tukey outliers of count means or variances removed", M1, ~tuk),
                     ("main, no removal", M1, None), ("all units, no removal", MA, None)):
    ffv, ffn, p2, p1, n, cm, cv = ff_test(M, keep)
    print(f"{lab}: n {n}; FF V {ffv:.3f}, NV {ffn:.3f}; permutation P two-sided {p2:.3f}, one-sided {p1:.3f}; "
          f"count mean range {cm.min():.1f} to {cm.max():.1f}, variance range {cv.min():.1f} to {cv.max():.1f}")
ffv, ffn, p2, p1, n, cm, cv = ff_test(M1, (~o1).values)
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5), sharex=True, sharey=True)
for i, (lab, f_) in enumerate((("V", ffv), ("NV", ffn))):
    ax[i].scatter(cm[:, i], cv[:, i], s=6)
    xx = np.linspace(0, cm.max(), 2)
    ax[i].plot(xx, f_ * xx, "k")
    ax[i].set_title(f"{lab}, FF {f_:.2f}")
    ax[i].set_xlabel("spike count mean")
ax[0].set_ylabel("spike count variance")
plt.tight_layout()
plt.show()
print("all points are within the axes (axes span the data).")

**Result: not reproduced (resting on an assumption).** With the Fig. 4 outliers removed
(reading (a), 356 cells), FF is 1.84 in V and 3.19 in NV, larger in NV but not significantly
(permutation P = 0.280) and both above tolerance. With Tukey outliers of the count means or
variances removed (reading (b), 295 cells), FF is 0.98 and 1.24, larger in NV with P = 0.020, but
both below tolerance. Reading (a) is adopted, as the paper defines outliers only for Fig. 4.

## cv2-larger-nv

> cv2-larger-nv: The squared coefficient of variation of interspike intervals (CV2) was also significantly larger in NV than in V (Kruskal-Wallis test, P < 0.05).

In [ ]:
def cv2(M):
    I = indexes(M)
    out = []
    for r, p in zip(M.itertuples(), I.pnv):
        row = {}
        for ci, lab in ((0, "V"), (1, "NV")):
            isis = [np.diff(s) for s in r.hold_spikes[ci, p]]
            pooled = np.concatenate(isis)
            row[lab + "_a"] = (pooled.std() / pooled.mean()) ** 2 if len(pooled) >= 2 else np.nan
            per = [(i_.std() / i_.mean()) ** 2 for i_ in isis if len(i_) >= 2]
            row[lab + "_b"] = np.mean(per) if per else np.nan
        out.append(row)
    return pd.DataFrame(out)
for lab, M in (("main", M1), ("all units", MA)):
    C = cv2(M)
    for rd in ("a", "b"):
        x = C[["V_" + rd, "NV_" + rd]].dropna()
        h, p = stats.kruskal(x["V_" + rd], x["NV_" + rd])
        print(f"{lab}, reading ({rd}): n {len(x)}; median CV2 V {x['V_' + rd].median():.3f}, NV {x['NV_' + rd].median():.3f}; "
              f"mean V {x['V_' + rd].mean():.3f}, NV {x['NV_' + rd].mean():.3f}; Kruskal-Wallis H {h:.2f}, P {p:.3g}; "
              f"dropped (fewer than two ISIs) {len(C) - len(x)}")

**Result: reproduced (resting on an assumption).** With the ISIs of a cell's five trials pooled
(reading (a)), CV2 is larger in NV (median 0.737 against 0.622; Kruskal-Wallis P = 0.024, 318
cells). Averaged over trials (reading (b)) the difference is in the same direction but not
significant (0.510 against 0.456, P = 0.098). Reading (a), one CV2 "for each cell and condition"
(Methods), is adopted. Which condition is NV rests on the code mapping.

## Decoding: fig6-decoding-vv, fig6-decoding-nvnv, decoding-v-better-than-nv, fig6-decoding-cross

> fig6-decoding-vv: A maximum-likelihood decoder trained and tested on V data decoded hand position with an average accuracy across targets of 91.6 +/- 9%.
>
> fig6-decoding-nvnv: Trained and tested on NV data, the decoder's accuracy fell to 81.9 +/- 18.0%.
>
> decoding-v-better-than-nv: Decoding accuracy was significantly higher in V than in NV (t-test, P < 0.05), a small but significant role of vision in representing limb position.
>
> fig6-decoding-cross: Training and testing on different conditions gave lower accuracies, 60.7 +/- 22% trained on NV and tested on V and 69.1 +/- 17% trained on V and tested on NV, so decoding degraded whenever NV data were used.

All units of the population are treated as simultaneously recorded, and the trials of different
units are joined by their order within target and condition (criterion reading (i)).

In [ ]:
def counts_of(M, key="hold_count"):
    return np.stack(M[key].values)  # [cell, cond, target, trial]
def acc_per_target(conf):
    return np.diag(conf) / conf.sum(axis=1) * 100
def run_decoder(C, tr, te, loo=True):
    return acc_per_target(ppc.decode(C[:, tr], C[:, te] if te != tr else C[:, tr], loo=loo)) if te != tr else \
        acc_per_target(ppc.decode(C[:, tr], C[:, tr]))
def boot(C, n_boot=200, size=None, seed=3, schemes=((0, 0), (1, 1), (1, 0), (0, 1)), loo=True):
    rng = np.random.default_rng(seed)
    n = C.shape[0]
    out = {s: [] for s in schemes}
    for b in range(n_boot):
        idx = rng.integers(0, n, size or n)
        Cb = C[idx]
        for s in schemes:
            out[s].append(run_decoder(Cb, s[0], s[1], loo))
    return {s: np.array(v) for s, v in out.items()}  # [boot, target]
names_s = {(0, 0): "V/V", (1, 1): "NV/NV", (1, 0): "train NV, test V", (0, 1): "train V, test NV"}
DEC = {}
for lab, M in (("main", M1), ("all units", MA)):
    C = counts_of(M)
    full = {s: run_decoder(C, *s) for s in names_s}
    print(f"{lab}: {C.shape[0]} cells, full population, accuracy per target and mean:")
    for s, a_ in full.items():
        print(f"   {names_s[s]:18s} {np.round(a_, 0)} mean {a_.mean():.1f}")
    for bl, size in (("(a) resample n cells", None), ("(b) populations of 200 cells", 200)):
        B = boot(C, size=size)
        DEC[(lab, bl)] = B
        for s in names_s:
            acc = B[s].mean(axis=1)
            print(f"   bootstrap {bl}, {names_s[s]:18s}: mean {acc.mean():.1f} +/- {acc.std(ddof=1):.1f} (SD over 200); "
                  f"SD over targets of the mean {B[s].mean(axis=0).std(ddof=1):.1f}")
    Bx = boot(C, schemes=((1, 0), (0, 1)), loo=False, seed=4)
    for s in ((1, 0), (0, 1)):
        print(f"   cross reading (b), all five trials: {names_s[s]} {Bx[s].mean():.1f} +/- {Bx[s].mean(axis=1).std(ddof=1):.1f}")
print("paper: V/V 91.6 +/- 9, NV/NV 81.9 +/- 18.0, train NV test V 60.7 +/- 22, train V test NV 69.1 +/- 17")

In [ ]:
for (lab, bl), B in DEC.items():
    v_t, nv_t = B[(0, 0)].mean(axis=0), B[(1, 1)].mean(axis=0)
    v_b, nv_b = B[(0, 0)].mean(axis=1), B[(1, 1)].mean(axis=1)
    print(f"{lab}, {bl}: V {v_b.mean():.1f}, NV {nv_b.mean():.1f}; (a) paired t over 8 targets P {stats.ttest_rel(v_t, nv_t).pvalue:.3g}; "
          f"(b) two-sample t over 200 bootstraps P {stats.ttest_ind(v_b, nv_b).pvalue:.3g}; (c) two-sample t over 8 targets P {stats.ttest_ind(v_t, nv_t).pvalue:.3g}")
Cf = counts_of(M1)[:, ::-1]
Bf = boot(Cf, schemes=((0, 0), (1, 1)))
print(f"other code mapping (V = 7), main, (a): V {Bf[(0, 0)].mean():.1f}, NV {Bf[(1, 1)].mean():.1f}")

**Result: not reproduced.** Trained and tested on V, the decoder's accuracy over 200 bootstrap
resamples of the 359 cells is 77.4 +/- 6.5% (reading (a)) and 73.3% with populations of 200 cells
(reading (b)), below 91.6 +/- 9%. The full population without resampling gives 87.5%.

**Result: not reproduced.** Trained and tested on NV, accuracy is 63.7 +/- 7.0% (reading (a)) and
57.2% (reading (b)), just below the 63.9% lower bound of 81.9 +/- 18.0%. The full population gives
77.5%.

**Result: reproduced (resting on an assumption).** V accuracy exceeds NV accuracy (77.4 against
63.7%) and every t-test reading finds it (paired over targets P = 0.0087, over bootstraps P <
1e-60, unpaired over targets P = 0.0081), as do populations of 200 cells. Which condition is V
rests on the code mapping; with all 483 units the paired test gives P = 0.29.

**Result: reproduced.** Trained on NV and tested on V, accuracy is 58.9% (paper 60.7 +/- 22);
trained on V and tested on NV, 52.7% (69.1 +/- 17); both are within tolerance and below the V/V
accuracy (77.4%). Training on all five trials (reading (b)) gives 62.1 and 56.8%. Swapping the code
mapping keeps both within tolerance and below the V/V accuracy (63.7%).

## decoding-subepochs

> decoding-subepochs: Decoding was less accurate with shorter windows, but did not differ between the first and the last 400 ms of the hold, in V or in NV.

In [ ]:
for lab, M in (("main", M1), ("all units", MA)):
    res = {}
    for w in ("hold_count", "first400_count", "last400_count"):
        res[w] = boot(counts_of(M, w), schemes=((0, 0), (1, 1)), seed=5)
    for ci, cn in ((0, "V"), (1, "NV")):
        s = (ci, ci)
        h, f_, l_ = (res[w][s] for w in ("hold_count", "first400_count", "last400_count"))
        print(f"{lab}, {cn}: 800 ms {h.mean():.1f}, first 400 {f_.mean():.1f}, last 400 {l_.mean():.1f}; first vs last: "
              f"(a) paired t over targets P {stats.ttest_rel(f_.mean(0), l_.mean(0)).pvalue:.3g}, "
              f"(b) two-sample t over bootstraps P {stats.ttest_ind(f_.mean(1), l_.mean(1)).pvalue:.3g}, "
              f"(c) two-sample t over targets P {stats.ttest_ind(f_.mean(0), l_.mean(0)).pvalue:.3g}; "
              f"difference {f_.mean() - l_.mean():.1f} points")

**Result: not reproduced.** The 400 ms windows decode less well than the 800 ms window (V 61.2 and
66.0 against 78.3%; NV 59.8 and 42.8 against 64.6%), as the paper says. In V the first and last 400
ms do not differ by the paired test (P = 0.38), but in NV the last 400 ms decode 17.0 points worse
than the first (paired P = 0.0073, unpaired over targets P = 0.017), so a difference is found.

## decoding-nonsignificant-cells

> decoding-nonsignificant-cells: Neurons with no ANOVA effect of position, vision or interaction still decoded position above chance at some targets, and slightly better in V than in NV (20% against 14%), though this difference was not significant.

In [ ]:
for lab, M in (("main", M1), ("all units", MA)):
    none = groups(M)["none"].values
    C = counts_of(M)[none]
    B = boot(C, schemes=((0, 0), (1, 1)), seed=6)
    v_t, nv_t = B[(0, 0)].mean(0), B[(1, 1)].mean(0)
    print(f"{lab}: {none.sum()} cells without ANOVA effects; V {v_t.mean():.1f}, NV {nv_t.mean():.1f} (chance 12.5); "
          f"per target V {np.round(v_t, 0)}, NV {np.round(nv_t, 0)}; targets above chance in both: {int(np.sum((v_t > 12.5) & (nv_t > 12.5)))}; "
          f"t-tests P (a) {stats.ttest_rel(v_t, nv_t).pvalue:.3g}, (b) {stats.ttest_ind(B[(0, 0)].mean(1), B[(1, 1)].mean(1)).pvalue:.3g}, "
          f"(c) {stats.ttest_ind(v_t, nv_t).pvalue:.3g}")
print("paper: V 20%, NV 14%")

**Result: not reproduced.** The 194 cells without ANOVA effects decode 24.7% in V and 12.5% in NV
(paper 20 and 14%; neither within tolerance), V above NV, with 4 targets above chance in both, but
the difference is significant (paired P = 0.00016), where the paper found it was not. All units (266
cells) give 18.3 and 17.6% (P = 0.85).

## eye-position-no-difference

> eye-position-no-difference: Neither mean eye position nor the variance of its horizontal or vertical component differed between V and NV for any target (MANOVA and Levene's test).

The claim is blocked (`metadata_insufficient`): the data set holds no documented eye position.
The analog files' channels are named only AD01 to AD16, and none behaves like a fixation trace
(not shown; read while preparing). The 24 field record (in part of the sessions) ends with two
numbers near (0, -1.8) that might be an eye position, but no file names them, and their spread
within a session (SD about 0.05 to 0.07) is well below the eye tracker's stated resolution of
0.25 degrees (Methods, Experimental apparatus), so they cannot be taken as single eye samples. The
test is run on them below only to show what they give; it decides nothing.

In [ ]:
E = pd.concat([t.assign(session=k) for k, t in kept.items() if "f22" in t and t.f22.notna().all()])
print("sessions with the two fields:", E.session.nunique(), "; trials:", len(E))
print("field 22 range", E.f22.min(), "to", E.f22.max(), "; field 23 range", E.f23.min(), "to", E.f23.max())
print("within-session SD (median over sessions): field 22", round(E.groupby("session").f22.std().median(), 3),
      "; field 23", round(E.groupby("session").f23.std().median(), 3))
rows = []
for tg in ppc.TARGETS:
    v = E[(E.target == tg) & (E.cond == ppc.V_CODE)][["f22", "f23"]].values
    nv = E[(E.target == tg) & (E.cond == ppc.NV_CODE)][["f22", "f23"]].values
    F, p = ppc.hotelling2(v, nv)
    rows.append(dict(target=ppc.TARGET_DEG[tg], n_V=len(v), n_NV=len(nv), mean_V=np.round(v.mean(0), 3), mean_NV=np.round(nv.mean(0), 3),
                     p_manova=p, p_levene_h=stats.levene(v[:, 0], nv[:, 0], center="mean").pvalue,
                     p_levene_v=stats.levene(v[:, 1], nv[:, 1], center="mean").pvalue,
                     p_levene_h_median=stats.levene(v[:, 0], nv[:, 0]).pvalue, p_levene_v_median=stats.levene(v[:, 1], nv[:, 1]).pvalue))
EY = pd.DataFrame(rows)
print("(a) per target, trials pooled over sessions:")
print(EY.to_string(index=False))
print("targets with any P < 0.05:", int(((EY.p_manova < 0.05) | (EY.p_levene_h < 0.05) | (EY.p_levene_v < 0.05)).sum()))
rows = []
for (k, tg), g in E.groupby(["session", "target"]):
    v = g[g.cond == ppc.V_CODE][["f22", "f23"]].values
    nv = g[g.cond == ppc.NV_CODE][["f22", "f23"]].values
    rows.append(dict(p_manova=ppc.hotelling2(v, nv)[1], p_h=stats.levene(v[:, 0], nv[:, 0], center="mean").pvalue,
                     p_v=stats.levene(v[:, 1], nv[:, 1], center="mean").pvalue))
EP = pd.DataFrame(rows)
print("(b) per session and target, % significant: MANOVA", round(100 * np.mean(EP.p_manova < 0.05), 1),
      "; Levene horizontal", round(100 * np.mean(EP.p_h < 0.05), 1), "; vertical", round(100 * np.mean(EP.p_v < 0.05), 1),
      "; undefined", int(EP.p_manova.isna().sum()))

**Result: blocked (metadata_insufficient).** The data set holds no documented eye position.

## aborted-trials-nv (blocked)

> aborted-trials-nv: In one animal, as the behavioral window around the reach targets was made smaller, aborted trials increased in both conditions, and were significantly more frequent on NV trials for smaller windows (t-test, P < 0.05), so the animals used vision to hold position.

The labels of V and NV are inferred above from the aborts themselves, so this claim cannot be
tested independently of that inference. The counts are shown for the record. Field 1 of the record
(8, 10 or 12) is read as the target window size; no file says what it is.

In [ ]:
a2 = a.copy()
a2["window"] = a2.params.apply(lambda p: p[1] if isinstance(p, tuple) else np.nan)
t_ = a2[a2.phase != "aborted before code 10"]
print(pd.crosstab([t_.window, t_.cond], t_.phase, margins=False).to_string())
print((t_.assign(ab=t_.phase == "aborted after code 10").groupby(["window", "cond"]).ab.mean() * 100).round(1).to_string())

**Result: blocked (metadata_insufficient).** Which code is NV was inferred above from the aborts
themselves, so the claim's comparison of NV with V aborts would test the inference, not the paper.
The window sizes are not documented either (field 1 of the record is read as one). For the record,
with field 1 at 8, 10 and 12, 31.4, 26.6 and 5.7% of code 7 trials and 13.9, 24.4 and 3.6% of code 4
trials that reached code 10 were aborted.

## longer-hold-no-difference (blocked)

> longer-hold-no-difference: In a subset of sessions with the minimum hold extended to 1,200 ms, neither behavior nor neural responses differed appreciably.

In [ ]:
rows = []
for k, s_ in D["sessions"].items():
    t = s_["trials"]
    if "params" not in t:
        continue
    c = t[t.completed & t.params.notna()]
    if len(c) == 0:
        continue
    f11 = c.params.apply(lambda p: p[11])
    rows.append(dict(session=k, field11=f11.iloc[0], field11_values=len(set(f11)), n=len(c), min_10_to_12=(c.t12 - c.t10).min()))
HP = pd.DataFrame(rows)
print("completed trials' record field 11 against the shortest code 10 to code 12 interval of each session (s):")
print(HP.groupby("field11").agg(sessions=("session", "size"), shortest_min=("min_10_to_12", "min"),
                                 shortest_median=("min_10_to_12", "median"), shortest_max=("min_10_to_12", "max")).round(3).to_string())
print("sessions whose field 11 changes within the session:", int((HP.field11_values > 1).sum()))
allp = allt[allt.params.notna()]
f11_all = allp.params.apply(lambda p: p[11])
print("trials with field 11 >= 1200, any outcome:", int((f11_all >= 1200).sum()), "; completed:", int(((f11_all >= 1200) & allp.completed).sum()))

**Result: blocked (data_missing).** Record field 11 follows the shortest interval from code 10
to code 12 in each session (800 to 1,100 ms against shortest intervals of 0.83 to 1.15 s), so it is
read as the minimum hold. No completed trial has a minimum hold of 1,200 ms or more; the data set
holds no subset of sessions with the extended hold.

## Summary

| Claim | Outcome |
|---|---|
| behavior-mean-position-sessions | not_reproduced |
| behavior-variability-sessions | not_reproduced |
| table1-endpoint-means | partially_reproduced |
| table1-endpoint-sd | not_reproduced |
| fig3-example-neuron | not_reproduced |
| movement-epoch-tuning | not_reproduced |
| hold-position-main-effect | reproduced |
| hold-vision-main-effect | reproduced |
| hold-interaction | reproduced |
| hold-subepochs | not_reproduced |
| hold-overall-effects | partially_reproduced |
| correlation-endpoint-datasets | reproduced |
| fig4-indx1-negative | partially_reproduced |
| fig4-indx1-means | partially_reproduced |
| indx1-tuning-shift | not_reproduced |
| indx2-means | not_reproduced |
| rate-variance-nv-higher | partially_reproduced |
| fig5-fano-factor | not_reproduced |
| cv2-larger-nv | reproduced |
| fig6-decoding-vv | not_reproduced |
| fig6-decoding-nvnv | not_reproduced |
| decoding-v-better-than-nv | reproduced |
| fig6-decoding-cross | reproduced |
| decoding-subepochs | not_reproduced |
| decoding-nonsignificant-cells | not_reproduced |
| eye-position-no-difference | blocked (metadata_insufficient) |
| aborted-trials-nv | blocked (metadata_insufficient) |
| longer-hold-no-difference | blocked (data_missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.**

- Only monkey X: the data set holds no session with monkey B's target layout. The paper pools 219
  monkey X and 124 monkey B neurons; here 359 monkey X sessions with the complete design are used,
  one unit from each (the first sorted unit), with all 483 sorted units as a variant. Counts are
  compared with the paper's as percentages, or scaled to 343.
- Which condition code is V is inferred from more aborts after code 10 on code 7 trials (assumed:
  4 = V, 7 = NV). The directional results (rate variance, FF, CV2, V better than NV decoding) rest
  on this.
- The meaning of the event codes is read from their timing: 12 as the end of the hold, 8 as
  movement onset (cursor leaving the start window).
- Each trial has one hand position (the record's x, y, z), not a position trace, so behavioural
  tests use it in place of the mean over the 800 ms hold. No continuous arm or eye position is
  identified in the data.
- Where a cell of the design has more than five completed trials (4 sessions), the first five are
  used.

**What was found about the data set.**

- The data descriptor says 343 cells, 219 from monkey X and 124 from monkey B; every session with a
  trial record has monkey X's layout, and there are more usable monkey X sessions (359, with 485
  sorted units) than the paper's 219. Which of them the paper used is not documented.
- The condition (V or NV) of a trial is coded as 4 or 7 in an undocumented text record; the event
  codes are not described (`EventCodes.txt` is a generic list that does not match them), nor are the
  record's fields (target, condition, hand position, possibly eye position, task parameters).
- The analog files' channels are named AD01 to AD16 without description.
- Nine early files (`d<n>.mat`) have a shorter record without positions and a hold below 800 ms.
- The `.mat` spike times are in milliseconds and the event times in seconds; the descriptor gives
  neither unit. They agree with the `.plx` file of the session checked.
